# 05: Visualization — Publication-Quality Figures

**Goal**: Create all final publication-quality figures.

This notebook generates:
1. **Main results heatmap** (F1 scores: models × relation types)
2. **Relation type clustering** (dendrogram showing relation similarity)
3. **Model comparison** (bar charts, violin plots)
4. **2D projections** (UMAP/t-SNE of embedding space)
5. **Probe weight analysis** (most diagnostic dimensions)

All figures saved as both .pdf (publication) and .png (presentation) at 300dpi.

## 1. Setup and Load Results

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist, squareform
from dotenv import load_dotenv
import os

# Setup
sys.path.insert(0, str(Path.cwd().parent))
load_dotenv()
CACHE_DIR = Path(os.getenv("CACHE_DIR", "../results/embeddings"))
RESULTS_DIR = Path("../results")

np.random.seed(42)
sns.set_style("whitegrid")
sns.set_palette("Set2")

# Configure matplotlib for publication
plt.rcParams["font.size"] = 11
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["legend.fontsize"] = 11
plt.rcParams["figure.figsize"] = (12, 8)

print(f"Results directory: {RESULTS_DIR}")
print(f"Figures will be saved to: {RESULTS_DIR / 'figures'}")

## 2. Load All Results Data

In [ ]:
from datasets import load_dataset

# Load probing results
results_main = pd.read_csv(RESULTS_DIR / "probing_results.csv")
gains_df = pd.read_csv(RESULTS_DIR / "probing_results_with_gains.csv")

print(f"Loaded probing results: {len(results_main)} relations")
print(f"Columns: {list(results_main.columns)}")

# Load embeddings and labels
print(f"\nLoading embeddings and dataset...")
embeddings_dict = {
    "SBERT": np.load(CACHE_DIR / "sbert_vsr_train.npy"),
    "CLIP Text": np.load(CACHE_DIR / "clip_text_vsr_train.npy"),
    "CLIP MM": np.load(CACHE_DIR / "clip_multimodal_vsr_train.npy"),
}

vsr = load_dataset("cambridgeltl/vsr_random")
train_data = vsr["train"]
relation_labels = np.array([ex["relation_type"] for ex in train_data])

print(f"Dataset: {len(train_data)} examples")

## 3. Main Results Heatmap (Models × Relations)

In [ ]:
# Prepare heatmap data - top relations by mean F1
model_columns = ["sbert_f1", "clip_text_f1", "clip_mm_f1"]
heatmap_data = results_main["relation"] + "_f1"].copy()

top_n = 25
top_relations = results_main.nlargest(top_n, "sbert_f1")["relation"].values
heatmap_subset = results_main[results_main["relation"].isin(top_relations)].set_index("relation")
heatmap_subset = heatmap_subset[model_columns]
heatmap_subset.columns = ["SBERT", "CLIP Text", "CLIP MM"]

# Create heatmap
fig, ax = plt.subplots(figsize=(9, 14))
sns.heatmap(
    heatmap_subset,
    annot=True,
    fmt=".3f",
    cmap="RdYlGn",
    vmin=0,
    vmax=1,
    cbar_kws={"label": "F1 Score"},
    ax=ax,
    linewidths=0.5,
    linecolor="gray"
)
ax.set_title(f"Probing Results: Spatial Relation Decodability\nTop {top_n} relations by F1", 
            fontsize=14, fontweight="bold", pad=20)
ax.set_xlabel("Model", fontsize=12)
ax.set_ylabel("Spatial Relation Type", fontsize=12)
plt.tight_layout()

# Save as both PDF and PNG
plt.savefig(RESULTS_DIR / "figures" / "05_main_heatmap.pdf", dpi=300, bbox_inches="tight")
plt.savefig(RESULTS_DIR / "figures" / "05_main_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: 05_main_heatmap.pdf/png")

## 4. Relation Type Clustering (Dendrogram by CLIP Similarity)

In [ ]:
# Compute relation-to-relation similarity using CLIP embeddings
# Idea: relations that are semantically similar should have CLIP embeddings that are similar

clip_mm_emb = embeddings_dict["CLIP MM"]

# Get mean embedding per relation type
relation_embeddings = {}
for relation in np.unique(relation_labels):
    indices = np.where(relation_labels == relation)[0]
    relation_embeddings[relation] = clip_mm_emb[indices].mean(axis=0)

# String to compute RDM
relation_names = sorted(list(relation_embeddings.keys()))
relation_embs = np.array([relation_embeddings[r] for r in relation_names])

# Compute pairwise distances
pdist_vals = pdist(relation_embs, metric="cosine")
linkage_matrix = linkage(pdist_vals, method="average")

# Plot dendrogram
fig, ax = plt.subplots(figsize=(14, 8))
dendrogram(
    linkage_matrix,
    labels=relation_names,
    ax=ax,
    leaf_font_size=10
)
ax.set_title("Relation Type Clustering (by CLIP MM Embedding Similarity)", 
            fontsize=14, fontweight="bold")
ax.set_xlabel("Spatial Relation Type", fontsize=12)
ax.set_ylabel("Distance (cosine dissimilarity)", fontsize=12)
plt.xticks(rotation=90)
plt.tight_layout()

plt.savefig(RESULTS_DIR / "figures" / "05_relation_dendrogram.pdf", dpi=300, bbox_inches="tight")
plt.savefig(RESULTS_DIR / "figures" / "05_relation_dendrogram.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: 05_relation_dendrogram.pdf/png")

## 5. Model Comparison — Performance Distribution

In [ ]:
# Violin plot of F1 distributions
data_for_plot = []
for col, model in [("sbert_f1", "SBERT"), ("clip_text_f1", "CLIP Text"), ("clip_mm_f1", "CLIP MM")]:
    for f1 in results_main[col]:
        data_for_plot.append({"Model": model, "F1 Score": f1})

df_plot = pd.DataFrame(data_for_plot)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Violin plot
sns.violinplot(data=df_plot, x="Model", y="F1 Score", ax=ax1, palette="Set2")
ax1.set_title("F1 Score Distribution across Relations", fontsize=13, fontweight="bold")
ax1.set_ylabel("F1 Score", fontsize=12)
ax1.set_ylim([0, 1])
ax1.grid(axis="y", alpha=0.3)

# Box plot with points
sns.boxplot(data=df_plot, x="Model", y="F1 Score", ax=ax2, palette="Set2")
sns.stripplot(data=df_plot, x="Model", y="F1 Score", ax=ax2, color="black", alpha=0.3, size=3)
ax2.set_title("F1 Score Quartiles and Outliers", fontsize=13, fontweight="bold")
ax2.set_ylabel("F1 Score", fontsize=12)
ax2.set_ylim([0, 1])
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "figures" / "05_model_comparison.pdf", dpi=300, bbox_inches="tight")
plt.savefig(RESULTS_DIR / "figures" / "05_model_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: 05_model_comparison.pdf/png")

## 6. CLIP Gains over SBERT

In [ ]:
# Visualize CLIP improvements as scatter
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# CLIP text vs SBERT
ax1.scatter(results_main["sbert_f1"], results_main["clip_text_f1"], alpha=0.6, s=50, color="steelblue")
max_val = max(results_main["sbert_f1"].max(), results_main["clip_text_f1"].max())
ax1.plot([0, max_val], [0, max_val], "k--", lw=2, alpha=0.5, label="No improvement")
ax1.set_xlim([0, max_val* 1.05])
ax1.set_ylim([0, max_val* 1.05])
ax1.set_xlabel("SBERT F1", fontsize=12)
ax1.set_ylabel("CLIP Text F1", fontsize=12)
ax1.set_title("CLIP Text vs SBERT\n(text-only comparison)", fontsize=13, fontweight="bold")
ax1.legend()
ax1.grid(alpha=0.3)

# CLIP MM vs SBERT
ax2.scatter(results_main["sbert_f1"], results_main["clip_mm_f1"], alpha=0.6, s=50, color="seagreen")
ax2.plot([0, max_val], [0, max_val], "k--", lw=2, alpha=0.5, label="No improvement")
ax2.set_xlim([0, max_val* 1.05])
ax2.set_ylim([0, max_val* 1.05])
ax2.set_xlabel("SBERT F1", fontsize=12)
ax2.set_ylabel("CLIP MM F1", fontsize=12)
ax2.set_title("CLIP Multimodal vs SBERT\n(with image access)", fontsize=13, fontweight="bold")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "figures" / "05_clip_gains.pdf", dpi=300, bbox_inches="tight")
plt.savefig(RESULTS_DIR / "figures" / "05_clip_gains.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: 05_clip_gains.pdf/png")

## 7. 2D Projections (UMAP of embeddings by relation type)

In [ ]:
# UMAP projections (optional if umap-learn installed)
try:
    import umap
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for idx, (model_name, embeddings) in enumerate(embeddings_dict.items()):
        # Subsample for speed (UMAP can be slow)
        sample_indices = np.random.choice(len(embeddings), min(2000, len(embeddings)), replace=False)
        emb_sample = embeddings[sample_indices]
        rel_sample = relation_labels[sample_indices]
        
        # UMAP
        reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
        embedding_2d = reducer.fit_transform(emb_sample)
        
        # Plot
        ax = axes[idx]
        unique_relations = np.unique(rel_sample)
        colors = plt.cm.tab20(np.linspace(0, 1, len(unique_relations)))
        
        for rel_idx, relation in enumerate(unique_relations):
            mask = rel_sample == relation
            ax.scatter(embedding_2d[mask, 0], embedding_2d[mask, 1], 
                      alpha=0.5, s=20, label=relation if rel_idx < 10 else None,
                      color=colors[rel_idx])
        
        ax.set_title(model_name, fontsize=13, fontweight="bold")
        ax.set_xlabel("UMAP 1", fontsize=11)
        ax.set_ylabel("UMAP 2", fontsize=11)
        ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "figures" / "05_umap_projections.pdf", dpi=300, bbox_inches="tight")
    plt.savefig(RESULTS_DIR / "figures" / "05_umap_projections.png", dpi=300, bbox_inches="tight")
    plt.show()
    
    print("Saved: 05_umap_projections.pdf/png")
except ImportError:
    print("UMAP not installed. Skipping 2D projection visualization.")
    print("To use: pip install umap-learn")

## 8. Summary Table

In [ ]:
# Create summary statistics table
print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)

summary_stats = []
for model in ["SBERT", "CLIP Text", "CLIP MM"]:
    col = model.lower().replace(" ", "_") + "_f1"
    f1_scores = results_main[col]
    summary_stats.append({
        "Model": model,
        "Mean F1": f1_scores.mean(),
        "Std F1": f1_scores.std(),
        "Min F1": f1_scores.min(),
        "Median F1": f1_scores.median(),
        "Max F1": f1_scores.max(),
        "N Relations": len(f1_scores)
    })

summary_df = pd.DataFrame(summary_stats)
print(summary_df.to_string(index=False))

# Save summary table
summary_df.to_csv(RESULTS_DIR / "summary_statistics.csv", index=False)
print(f"\nSaved: summary_statistics.csv")

print("\n" + "="*80)
print("VISUALIZATIONS COMPLETE")
print("="*80)
print(f"\nAll figures saved to: {RESULTS_DIR / 'figures'}")
print(f"\nGenerated files:")
for fig_file in sorted((RESULTS_DIR / "figures").glob("05_*.png")):
    print(f"  ✓ {fig_file.name}")